# BÀI 4: ĐỊNH DẠNG FILE EXCEL XUẤT BẢN NÂNG CAO VỚI OPENPYXL
## Mục tiêu học tập:
1. Kết hợp sức mạnh tính toán của **Pandas** với khả năng trang trí báo cáo chuyên nghiệp của **OpenPyXL**.
2. Định dạng Header chuẩn thương hiệu (màu nền, màu chữ trắng, căn giữa, in đậm).
3. Tự động kẻ bảng viền mỏng (Thin Border) cho toàn bộ vùng dữ liệu.
4. Định dạng hiển thị số tiền có dấu phân cách hàng nghìn (`#,##0 VNĐ`) và ngày tháng (`dd/mm/yyyy`).
5. Thuật toán tự động căn chỉnh độ rộng cột vừa khít nội dung (**Auto-fit Column Width**).

### Ô 1: Chuẩn bị dữ liệu mẫu và ghi ra Excel với `pd.ExcelWriter`
> **Nguyên lý:** Pandas ghi thô dữ liệu vào file trước, sau đó `openpyxl` can thiệp vào bộ nhớ để trang trí từng ô tính.

In [ ]:
import pandas as pd
from datetime import datetime

# Tạo bảng dữ liệu tài sản hoàn chỉnh sau khi đã merge ở Bài 3
data_report = {
    "Mã tài sản": ["TS-2026-001", "TS-2026-002", "TS-2026-003", "TS-2026-004", "TS-2026-005"],
    "Tên tài sản": ["Máy tính xách tay Dell Latitude 5440", "Máy in đa năng HP LaserJet Pro", "Máy Photocopy Ricoh Aficio", "Máy Scan màu Epson V39", "Bộ lưu điện UPS APC 1500VA"],
    "Phòng ban quản lý": ["Phòng Kế toán", "Phòng Tín dụng", "Phòng Hành chính - Quản trị", "Phòng Khách hàng DN", "Phòng Quản lý rủi ro"],
    "Ngày kiểm kê": [datetime(2026, 3, 15), datetime(2026, 3, 16), datetime(2026, 3, 16), datetime(2026, 3, 18), datetime(2026, 3, 20)],
    "Số lượng": [5, 2, 1, 2, 4],
    "Nguyên giá": [21500000, 8900000, 52000000, 3400000, 11200000]
}

df = pd.DataFrame(data_report)
df["Thành tiền"] = df["Số lượng"] * df["Nguyên giá"]

excel_file = "Bao_cao_kiem_ke_tai_san_dinh_dang.xlsx"
writer = pd.ExcelWriter(excel_file, engine="openpyxl")
df.to_excel(writer, sheet_name="KiemKe_2026", index=False, startrow=2)  # Để trống 2 dòng đầu cho tiêu đề lớn

wb = writer.book
ws = writer.sheets["KiemKe_2026"]
print("Đã ghi dữ liệu nền tảng vào file Excel thành công!")

### Ô 2: Thiết kế tiêu đề báo cáo lớn và Header bảng
- Ghép ô (Merge cells) tiêu đề lớn từ A1 đến G1.
- Header bảng dòng 3: Màu xanh ngọc lục bảo (`00685E`), chữ trắng in đậm, căn giữa.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# 1. Tiêu đề lớn của báo cáo
ws.merge_cells("A1:G1")
title_cell = ws["A1"]
title_cell.value = "BÁO CÁO TỔNG HỢP KIỂM KÊ TÀI SẢN NĂM 2026"
title_cell.font = Font(name="Arial", size=14, bold=True, color="004B49")
title_cell.alignment = Alignment(horizontal="center", vertical="center")
ws.row_dimensions[1].height = 35

# 2. Trang trí Header bảng (Dòng 3)
header_fill = PatternFill(start_color="00685E", end_color="00685E", fill_type="solid")
header_font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
header_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

ws.row_dimensions[3].height = 28
for col_idx in range(1, len(df.columns) + 1):
    cell = ws.cell(row=3, column=col_idx)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = header_alignment

print("Đã thiết kế tiêu đề và header bảng thành công!")

### Ô 3: Kẻ viền (Border) và định dạng hiển thị số/tiền tệ/ngày tháng
- Kẻ viền mảnh (Thin Border) cho mọi ô từ dòng 3 đến dòng dữ liệu cuối cùng.
- Định dạng cột 'Ngày kiểm kê' theo dạng `dd/mm/yyyy`.
- Định dạng 'Nguyên giá' và 'Thành tiền' chuẩn dấu chấm phẩy và đơn vị tiền tệ (`#,##0 VNĐ`).

In [ ]:
thin_border_side = Side(border_style="thin", color="BFBFBF")
data_border = Border(
    left=thin_border_side,
    right=thin_border_side,
    top=thin_border_side,
    bottom=thin_border_side
)

start_row = 3
end_row = 3 + len(df)

for row in range(start_row, end_row + 1):
    ws.row_dimensions[row].height = 22
    for col in range(1, len(df.columns) + 1):
        cell = ws.cell(row=row, column=col)
        cell.border = data_border
        
        # Chỉ định dạng số liệu từ dòng 4 trở đi (trừ dòng header)
        if row > 3:
            # Cột 1 (Mã): Căn giữa
            if col == 1:
                cell.alignment = Alignment(horizontal="center", vertical="center")
            # Cột 4 (Ngày kiểm kê): dd/mm/yyyy
            elif col == 4:
                cell.number_format = "dd/mm/yyyy"
                cell.alignment = Alignment(horizontal="center", vertical="center")
            # Cột 5 (Số lượng): Căn giữa, định dạng số nguyên
            elif col == 5:
                cell.number_format = "#,##0"
                cell.alignment = Alignment(horizontal="center", vertical="center")
            # Cột 6 & 7 (Nguyên giá, Thành tiền): Định dạng số tiền tệ
            elif col in [6, 7]:
                cell.number_format = '#,##0" VNĐ"'
                cell.alignment = Alignment(horizontal="right", vertical="center")

print("Đã kẻ border và định dạng format số/ngày thành công!")

### Ô 4: Thêm dòng Tổng cộng (Grand Total) với công thức Excel tự động
- Không dùng Python tính sẵn số chết, mà chèn thẳng công thức `=SUM(...)` để file Excel tương tác động.

In [ ]:
total_row = end_row + 1
ws.cell(row=total_row, column=2, value="TỔNG CỘNG").font = Font(name="Arial", size=10, bold=True)
ws.cell(row=total_row, column=2).alignment = Alignment(horizontal="right", vertical="center")

# Chèn công thức SUM tự động cho cột Số lượng (E) và Thành tiền (G)
ws.cell(row=total_row, column=5, value=f"=SUM(E4:E{end_row})")
ws.cell(row=total_row, column=7, value=f"=SUM(G4:G{end_row})")

total_fill = PatternFill(start_color="E8F5E9", end_color="E8F5E9", fill_type="solid")
double_bottom_border = Border(
    top=Side(style="thin", color="000000"),
    bottom=Side(style="double", color="000000")
)

for col in range(1, len(df.columns) + 1):
    cell = ws.cell(row=total_row, column=col)
    cell.fill = total_fill
    cell.border = double_bottom_border
    cell.font = Font(name="Arial", size=10, bold=True)
    if col == 5:
        cell.number_format = "#,##0"
        cell.alignment = Alignment(horizontal="center", vertical="center")
    elif col == 7:
        cell.number_format = '#,##0" VNĐ"'
        cell.alignment = Alignment(horizontal="right", vertical="center")

print("Đã thêm dòng tổng cộng với công thức SUM động!")

### Ô 5: Tự động căn chỉnh độ rộng cột (Auto-fit Columns) & Đóng file hoàn tất
- Thuật toán tính toán độ dài ký tự dài nhất của từng cột và cộng thêm khoảng đệm để không bao giờ bị lỗi `###`.

In [ ]:
for col in ws.columns:
    max_len = 0
    col_letter = get_column_letter(col[0].column)
    # Duyệt từ dòng 3 (bỏ dòng tiêu đề A1 đã merge để không bị kéo giãn cột quá đà)
    for cell in col[2:]:
        if cell.value:
            # Xử lý chuỗi hiển thị tương đối để tính kích thước
            val_str = str(cell.value)
            max_len = max(max_len, len(val_str))
    ws.column_dimensions[col_letter].width = max(max_len + 5, 12)

# Lưu file hoàn tất
writer.close()
print(f"Xuất file thành công: {excel_file}")
print("Báo cáo đã sẵn sàng gửi ban giám đốc với đầy đủ định dạng chuẩn!")